In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
from datetime import date

import climato.config as config
from climato.sources.meteo_france import daily

In [3]:
DATA_DIR = config.get_data_dir()
RAW_DATA_DIR = config.get_raw_data_dir()
MANIFEST_DIR = DATA_DIR / "metadata" / "manifests"
MANIFEST_PATH = MANIFEST_DIR / "daily_files.parquet"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DAILY_DIR = PROCESSED_DIR / daily.MF_DAILY_RELATIVE_PATH

In [4]:
lf_source = pl.scan_parquet(
    PROCESSED_DAILY_DIR / "parameter_family=rr_t_vent",
    hive_partitioning=True,
    hive_schema={
        "dataset_year": pl.UInt16,
        "department": pl.String,
        "collection": pl.String,
        "period": pl.String,
    },
)

In [26]:
lf_test = (
    lf_source
    .filter(
        pl.col("collection") == "quot",
        pl.col("tntxm").is_not_null(),
        pl.col("department").str.to_integer().is_between(79, 79),
        pl.any_horizontal(
            pl.col("obs_date").is_between(date(1990, 12, 1), date(1990, 12, 5)),
            pl.col("obs_date").is_between(date(1991, 1, 4), date(1991, 1, 7)),
            pl.col("obs_date").is_between(date(1991, 1, 10), date(1991, 1, 13)),
            pl.col("obs_date").is_between(date(2020, 12, 27), date(2020, 12, 29)),
            pl.col("obs_date").is_between(date(2021, 1, 4), date(2021, 1, 6)),
        )
        # pl.col("period") == "latest",
    )
    .select(
        "station_id",
        "station_name",
        "obs_date",
        "tntxm",
        pl.col("obs_date").dt.epoch("d").alias("epoch_d"),
        pl.col("obs_date").dt.epoch("d").shift().alias("previous_epoch_d"),
        pl.col("station_id").shift().alias("previous_station_id"),
    )
    .with_columns(
        (
            pl.when(pl.col("station_id") != pl.col("previous_station_id"))
            .then(None)
            .when(pl.col("obs_date") >= date(2021, 1, 1))
            .then(pl.lit(date(2021, 1, 1)).dt.epoch("d"))
            .otherwise(pl.col("epoch_d"))
            .alias("obs_date_epoch_ref")
        ),
        (
            pl.when(pl.col("station_id") != pl.col("previous_station_id"))
            .then(None)
            .when(pl.col("previous_epoch_d") <= pl.lit(date(1990, 12, 31)).dt.epoch("d"))
            .then(pl.lit(date(1990, 12, 31)).dt.epoch("d"))
            .otherwise(pl.col("previous_epoch_d"))
            .alias("previous_date_epoch_ref")
        ),
    )
    .with_columns(
        (pl.col("epoch_d") - pl.col("previous_epoch_d") - 1).alias("gap_days"),
        pl.max_horizontal(0, (pl.col("obs_date_epoch_ref") - pl.col("previous_date_epoch_ref") - 1)).alias("gap_days_ref"),
    )
    .head(100)
)

with pl.Config(tbl_cols=11, tbl_rows=100, tbl_width_chars=300):
    print(lf_test.collect())

shape: (100, 11)
┌────────────┬───────────────────┬────────────┬───────┬─────────┬──────────────────┬─────────────────────┬────────────────────┬─────────────────────────┬──────────┬──────────────┐
│ station_id ┆ station_name      ┆ obs_date   ┆ tntxm ┆ epoch_d ┆ previous_epoch_d ┆ previous_station_id ┆ obs_date_epoch_ref ┆ previous_date_epoch_ref ┆ gap_days ┆ gap_days_ref │
│ ---        ┆ ---               ┆ ---        ┆ ---   ┆ ---     ┆ ---              ┆ ---                 ┆ ---                ┆ ---                     ┆ ---      ┆ ---          │
│ str        ┆ str               ┆ date       ┆ f64   ┆ i32     ┆ i32              ┆ str                 ┆ i32                ┆ i32                     ┆ i32      ┆ i32          │
╞════════════╪═══════════════════╪════════════╪═══════╪═════════╪══════════════════╪═════════════════════╪════════════════════╪═════════════════════════╪══════════╪══════════════╡
│ 79017001   ┆ NUEIL-LES-AUBIERS ┆ 1990-12-01 ┆ 2.6   ┆ 7639    ┆ null             

In [27]:
lf_test_dt = (
    lf_source
    .filter(
        pl.col("collection") == "quot",
        pl.col("tntxm").is_not_null(),
        pl.col("department").str.to_integer().is_between(79, 79),
        pl.any_horizontal(
            pl.col("obs_date").is_between(date(1990, 12, 1), date(1990, 12, 5)),
            pl.col("obs_date").is_between(date(1991, 1, 4), date(1991, 1, 7)),
            pl.col("obs_date").is_between(date(1991, 1, 10), date(1991, 1, 13)),
            pl.col("obs_date").is_between(date(2020, 12, 27), date(2020, 12, 29)),
            pl.col("obs_date").is_between(date(2021, 1, 4), date(2021, 1, 6)),
        )
        # pl.col("period") == "latest",
    )
    .select(
        "station_id",
        "station_name",
        "obs_date",
        "tntxm",
        pl.col("obs_date").shift(1).over("station_id").alias("previous_date"),
    )
    .with_columns(
        (
            pl.when(pl.col("obs_date") >= date(2021, 1, 1))
            .then(pl.lit(date(2021, 1, 1)))
            .otherwise(pl.col("obs_date"))
            .alias("obs_date_ref")
        ),
        (
            pl.when(pl.col("previous_date") <= pl.lit(date(1990, 12, 31)))
            .then(pl.lit(date(1990, 12, 31)))
            .otherwise(pl.col("previous_date"))
            .alias("previous_date_ref")
        ),
    )
    .with_columns(
        ((pl.col("obs_date") - pl.col("previous_date")).dt.total_days() - 1).alias("gap_days"),
        pl.max_horizontal(0, ((pl.col("obs_date_ref") - pl.col("previous_date_ref")).dt.total_days() - 1)).alias("gap_days_ref"),
    )
    .head(100)
)

with pl.Config(tbl_cols=11, tbl_rows=100, tbl_width_chars=300):
    print(lf_test_dt.collect())

shape: (100, 9)
┌────────────┬───────────────────┬────────────┬───────┬───────────────┬──────────────┬───────────────────┬──────────┬──────────────┐
│ station_id ┆ station_name      ┆ obs_date   ┆ tntxm ┆ previous_date ┆ obs_date_ref ┆ previous_date_ref ┆ gap_days ┆ gap_days_ref │
│ ---        ┆ ---               ┆ ---        ┆ ---   ┆ ---           ┆ ---          ┆ ---               ┆ ---      ┆ ---          │
│ str        ┆ str               ┆ date       ┆ f64   ┆ date          ┆ date         ┆ date              ┆ i64      ┆ i64          │
╞════════════╪═══════════════════╪════════════╪═══════╪═══════════════╪══════════════╪═══════════════════╪══════════╪══════════════╡
│ 79017001   ┆ NUEIL-LES-AUBIERS ┆ 1990-12-01 ┆ 2.6   ┆ null          ┆ 1990-12-01   ┆ null              ┆ null     ┆ 0            │
│ 79017001   ┆ NUEIL-LES-AUBIERS ┆ 1990-12-02 ┆ 1.8   ┆ 1990-12-01    ┆ 1990-12-02   ┆ 1990-12-31        ┆ 0        ┆ 0            │
│ 79017001   ┆ NUEIL-LES-AUBIERS ┆ 1990-12-03 ┆ 1.2  

In [57]:
from climato.utils.datetime import utc_now_iso


lf_test_dt = (
    lf_source
    .filter(
        pl.col("collection") == "quot",
        pl.col("tntxm").is_not_null(),
        pl.col("department").str.to_integer().is_between(79, 79),
        pl.any_horizontal(
            pl.col("obs_date").is_between(date(1990, 12, 1), date(1990, 12, 5)),
            pl.col("obs_date").is_between(date(1991, 1, 4), date(1991, 1, 7)),
            pl.col("obs_date").is_between(date(1991, 1, 10), date(1991, 1, 13)),
            pl.col("obs_date").is_between(date(2020, 12, 27), date(2020, 12, 29)),
            pl.col("obs_date").is_between(date(2021, 1, 4), date(2021, 1, 6)),
        )
        # pl.col("period") == "latest",
    )
    .select(
        "station_id",
        "station_name",
        "obs_date",
        "tntxm",
        # pl.col("obs_date").shift(1).over("station_id").alias("previous_date"),
    )
    .sort(["station_id", "obs_date"])
)

date_diff = (
    pl.col("obs_date").diff().over("station_id")
)

previous_obs_date = pl.col("obs_date") - date_diff

gap_days = date_diff.dt.total_days() - 1

REF_START = date(1991, 1, 1)
REF_END = date(2020, 12, 31)
START_1991 = date(1991, 1, 1)
START_1975 = date(1975, 1, 1)
START_1950 = date(1950, 1, 1)
RECENT_END = date(2026, 9, 30)

gap_start = previous_obs_date + pl.duration(days=1)
gap_end = pl.col("obs_date") - pl.duration(days=1)

gap_start_ref = (
    pl.when(gap_start < REF_START)
    .then(pl.lit(REF_START))
    .otherwise(gap_start)
)

gap_end_ref = (
    pl.when(gap_end > REF_END)
    .then(pl.lit(REF_END))
    .otherwise(gap_end)
)

# Gaps that fall entirely outside the reference period will give 
internal_gap_days_ref = (
    pl.max_horizontal(
        0,
        (gap_end_ref - gap_start_ref).dt.total_days() + 1,
    )
)

is_ref = pl.col("obs_date").is_between(REF_START, REF_END)
since_1991 = pl.col("obs_date").is_between(START_1991, RECENT_END)
since_1975 = pl.col("obs_date").is_between(START_1975, RECENT_END)
since_1950 = pl.col("obs_date").is_between(START_1950, RECENT_END)

lf_test_dt = (
    lf_source
    .filter(
        pl.col("collection") == "quot",
        pl.col("tntxm").is_not_null(),
        # pl.col("department").str.to_integer().is_between(79, 79),
        # pl.any_horizontal(
        #     pl.col("obs_date").is_between(date(1990, 12, 1), date(1990, 12, 5)),
        #     pl.col("obs_date").is_between(date(1991, 1, 4), date(1991, 1, 7)),
        #     pl.col("obs_date").is_between(date(1991, 1, 10), date(1991, 1, 13)),
        #     pl.col("obs_date").is_between(date(2020, 12, 27), date(2020, 12, 29)),
        #     pl.col("obs_date").is_between(date(2021, 1, 4), date(2021, 1, 6)),
        # )
        # pl.col("period") == "latest",
    )
    .select(
        "station_id",
        "station_name",
        "obs_date",
        "tntxm",
        # pl.col("obs_date").shift(1).over("station_id").alias("previous_date"),
    )
    .sort(["station_id", "obs_date"])
    .with_columns(
        # previous_obs_date.alias("previous_obs_date"),
        # gap_days.alias("gap_days"),
        # gap_start.alias("gap_start"),
        # gap_end.alias("gap_end"),
        # gap_start_ref.alias("gap_start_ref"),
        # gap_end_ref.alias("gap_end_ref"),
        gap_days.alias("gap_days"),
        internal_gap_days_ref.alias("internal_gap_days_ref"),
    )
    .group_by("station_id", "station_name")
    .agg(
        # History
        n_obs=pl.len(),
        first_obs=pl.col("obs_date").min(),
        last_obs=pl.col("obs_date").max(),
        max_gap_days=pl.col("gap_days").max(),
        n_gaps=(pl.col("gap_days") > 0).sum(),
        n_gaps_gt_7d=(pl.col("gap_days") > 7).sum(),
        n_gaps_gt_30d=(pl.col("gap_days") > 30).sum(),

        # 1991
        n_obs_since_1991=since_1991.sum(),

        # 1975
        n_obs_since_1975=since_1975.sum(),

        # 1950
        n_obs_since_1950=since_1950.sum(),

        # Ref
        n_obs_ref=is_ref.sum(),
        first_obs_ref=pl.col("obs_date").filter(is_ref).min(),
        last_obs_ref=pl.col("obs_date").filter(is_ref).max(),
        max_internal_gap_days_ref=pl.col("internal_gap_days_ref").max(),
    )
    .with_columns(
        (
            pl.col("first_obs_ref") - pl.lit(REF_START)
        )
        .dt.total_days()
        .alias("leading_gap_days_ref"),
        (
            pl.lit(REF_END) - pl.col("last_obs_ref")
        )
        .dt.total_days()
        .alias("trailing_gap_days_ref"),
    )
    .with_columns(
        pl.max_horizontal(
            pl.col("max_internal_gap_days_ref").fill_null(0),
            pl.col("leading_gap_days_ref"),
            pl.col("trailing_gap_days_ref"),
        )
        .alias("max_gap_days_ref"),
    )
    .with_columns(
        ((pl.col("last_obs") - pl.col("first_obs")).dt.total_days() + 1).alias("span_days"),
        pl.lit((REF_END - REF_START).days + 1).alias("span_days_ref"),
        ((pl.lit(RECENT_END) - pl.lit(START_1950)).dt.total_days() + 1).alias("span_days_since_1950"),
        ((pl.lit(RECENT_END) - pl.lit(START_1975)).dt.total_days() + 1).alias("span_days_since_1975"),
        ((pl.lit(RECENT_END) - pl.lit(START_1991)).dt.total_days() + 1).alias("span_days_since_1991"),
        pl.lit("tntxm").alias("parameter"),
    )
    .with_columns(
        (pl.col("n_obs") / pl.col("span_days") * 100).alias("coverage_active_span"),
        (pl.col("n_obs_ref") / pl.col("span_days_ref") * 100).alias("coverage_span_ref"),
        (pl.col("n_obs_since_1950") / pl.col("span_days_since_1950") * 100).alias("coverage_span_1950"),
        (pl.col("n_obs_since_1975") / pl.col("span_days_since_1975") * 100).alias("coverage_span_1975"),
        (pl.col("n_obs_since_1991") / pl.col("span_days_since_1991") * 100).alias("coverage_span_1991"),
    )
    .select(
        "station_id", "station_name", "parameter",
        "first_obs", "last_obs", "n_obs", "span_days", "coverage_active_span", "n_gaps", "n_gaps_gt_7d", "n_gaps_gt_30d", "max_gap_days",
        "first_obs_ref", "last_obs_ref", "n_obs_ref", "span_days_ref", "coverage_span_ref", "max_gap_days_ref",
        "n_obs_since_1950", "span_days_since_1950", "coverage_span_1950",
        "n_obs_since_1975", "span_days_since_1975", "coverage_span_1975",
        "n_obs_since_1991", "span_days_since_1991", "coverage_span_1991",
    )
    .filter(
        # pl.col("first_obs").dt.date().le(date(1991, 1, 1)),
        # pl.col("last_obs").dt.date().ge(date(2020, 12, 31)),
        pl.col("last_obs") >= date(2026, 10, 1),
        pl.col("coverage_span_1975") >= 99,
    )
    .sort("station_id")
)

df_test_dt = lf_test_dt.collect()
with pl.Config(tbl_cols=30, tbl_rows=100, tbl_width_chars=300):
    # print(lf_test_dt.explain(optimized=True))
    print(df_test_dt)
    df_test_dt.write_csv(
        config.PROJECT_ROOT / "dump" / f"stations_candidates_{utc_now_iso()}.csv",
        separator=";",
    )

shape: (333, 27)
┌───────────┬───────────┬───────────┬───────────┬───────────┬───────┬───────────┬───────────┬────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ station_i ┆ station_n ┆ parameter ┆ first_obs ┆ last_obs  ┆ n_obs ┆ span_days ┆ coverage_ ┆ n_gaps ┆ n_gaps_g ┆ n_gaps_g ┆ max_gap_ ┆ first_ob ┆ last_obs ┆ n_obs_re ┆ span_day ┆ coverage ┆ max_gap_ ┆ n_obs_si ┆ span_day ┆ coverage ┆ n_obs_si ┆ span_day ┆ coverage ┆ n_obs_si ┆ span_day ┆ coverage │
│ d         ┆ ame       ┆ ---       ┆ ---       ┆ ---       ┆ ---   ┆ ---       ┆ active_sp ┆ ---    ┆ t_7d     ┆ t_30d    ┆ days     ┆ s_ref    ┆ _ref     ┆ f        ┆ s_ref    ┆ _span_re ┆ days_ref ┆ nce_1950 ┆ s_since_ ┆ _span_19 ┆ nce_1975 ┆ s_since_ ┆ _span_19 ┆ nce_1991 ┆ s_since_ ┆ _span_19 │
│ ---       ┆ ---       ┆ str       ┆ date      ┆ date      ┆ u32   ┆ i64       

In [ ]:
lf_tntxm = (
    lf_source
    .filter(
        pl.col("collection") == "quot",
        pl.col("tntxm").is_not_null(),
        # pl.col("department").str.to_integer().is_between(70, 79),
        # pl.col("period") == "latest",
    )
    .select(
        "station_id",
        "station_name",
        "obs_date",
        "tntxm",
        pl.col("obs_date").dt.epoch("d").alias("epoch"),
        pl.col("obs_date").dt.epoch("d").shift().alias("previous_epoch"),
        # (
        #     pl.when(pl.col("obs_date") >= date(2020, 12, 31))
        #     .then(pl.lit(date(2020, 12, 31)))
        #     .otherwise(pl.col("obs_date"))
        #     .alias("obs_date_ref")
        # ),
        # (
        #     pl.when(pl.col("obs_date").shift() <= date(1991, 1, 1))
        #     .then(pl.lit(date(1991, 1, 1)))
        #     .otherwise(pl.col("obs_date").shift())
        #     .alias("previous_date_ref")
        # ),
    )
    # .with_columns(
    #     pl.col("obs_date").shift().alias("previous_date"),
    # )
    .with_columns(
        (
            pl.when(pl.col("obs_date") >= date(2020, 12, 31))
            .then(pl.lit(date(2020, 12, 31)).dt.epoch("d"))
            .otherwise(pl.col("epoch"))
            .alias("obs_date_epoch_ref")
        ),
        (
            pl.when(pl.col("previous_epoch") <= pl.lit(date(1991, 1, 1)).dt.epoch("d"))
            .then(pl.lit(date(1991, 1, 1)).dt.epoch("d"))
            .otherwise(pl.col("previous_epoch"))
            .alias("previous_date_epoch_ref")
        ),
    )
    .group_by("station_id", "station_name")
    .agg(
        n_obs=pl.len(),
        first_obs=pl.col("obs_date").min(),
        last_obs=pl.col("obs_date").max(),
        n_obs_ref=pl.all_horizontal(
            pl.col("tntxm").is_not_null(),
            pl.col("obs_date").is_between(date(1991, 1, 1), date(2020, 12, 31)),
        ).sum(),
        max_gap=pl.col("epoch").diff().max() - 1,
        max_gap_ref=(pl.col("previous_date_epoch_ref") - pl.col("obs_date_epoch_ref") + 1).max() - 1,
    )
    .with_columns(
        ((pl.col("last_obs") - pl.col("first_obs")).dt.total_days() + 1).alias("span_days"),
        pl.lit("tntxm").alias("parameter"),
    )
    .with_columns(
        (pl.col("n_obs") / pl.col("span_days") * 100).alias("coverage_active_span"),
        (pl.col("n_obs_ref") / ((date(2020, 12, 31) - date(1991, 1, 1)).days + 1) * 100).alias("coverage_span_ref"),
    )
    .filter(
        pl.col("first_obs").dt.date().le(date(1991, 1, 1)),
        pl.col("last_obs").dt.date().ge(date(2020, 12, 31)),
        pl.col("coverage_active_span") >= 90,
    )
    .sort("station_id")
)

print(lf_tntxm)
print()
print(lf_tntxm.explain(optimized=True))

df_tntxm = lf_tntxm.collect(engine="streaming")

with pl.Config(tbl_rows=100, tbl_width_chars=200):
    print(df_tntxm)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SORT BY [col("station_id")]
  FILTER col("coverage_active_span") >= 90.0
  FROM
    FILTER (col("last_obs").dt.date() >= 2020-12-31)
    FROM
      FILTER (col("first_obs").dt.date() <= 1991-01-01)
      FROM
         WITH_COLUMNS:
         [((col("n_obs").cast(Int64) / col("span_days")) * 100.0).alias("coverage_active_span"), ((col("n_obs_ref") / 10958) * 100.0).alias("coverage_span_ref")] 
           WITH_COLUMNS:
           [(((col("last_obs") - col("first_obs"))).dt.total_days() + 1).alias("span_days"), "tntxm".alias("parameter")] 
            AGGREGATE[maintain_order: false]
              [len().alias("n_obs"), col("obs_date").min().alias("first_obs"), col("obs_date").max().alias("last_obs"), (col("tntxm").is_not_null() & col("obs_date").is_between([1991-01-01, 2020-12-31])).sum().alias("n_obs_ref"), (col("epoch").diff([dyn int: 1]).max() - 1).alias("max_gap"), (((col("previous_date_epoch_ref") - col("o

In [10]:
with pl.Config(tbl_rows=100, tbl_width_chars=200):
    print(df_tntxm)

shape: (801, 8)
┌────────────┬─────────────────────────────┬───────┬────────────┬────────────┬───────────┬───────────┬──────────────────────┐
│ station_id ┆ station_name                ┆ n_obs ┆ first_obs  ┆ last_obs   ┆ span_days ┆ parameter ┆ coverage_active_span │
│ ---        ┆ ---                         ┆ ---   ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---                  │
│ str        ┆ str                         ┆ u32   ┆ date       ┆ date       ┆ i64       ┆ str       ┆ f64                  │
╞════════════╪═════════════════════════════╪═══════╪════════════╪════════════╪═══════════╪═══════════╪══════════════════════╡
│ 01028001   ┆ BANEINS                     ┆ 12389 ┆ 1987-01-01 ┆ 2020-12-31 ┆ 12419     ┆ tntxm     ┆ 99.758435            │
│ 01037001   ┆ BENONCES                    ┆ 16285 ┆ 1976-06-01 ┆ 2020-12-31 ┆ 16285     ┆ tntxm     ┆ 100.0                │
│ 01089001   ┆ AMBERIEU                    ┆ 31232 ┆ 1941-02-01 ┆ 2026-10-13 ┆ 31301     ┆ tntxm     ┆

#### Parameter norms

In [69]:
param = "tx"

is_ref = pl.col("obs_date").is_between(REF_START, REF_END)

lf_norm_test = (
    lf_source
    .filter(
        pl.col("department") == "04",
        pl.col("station_id") == "04039001",
        # pl.col("obs_date").is_between(REF_START, REF_END),
    )
    .with_columns(
        pl.col("obs_date").dt.month().alias("month"),
        pl.col("obs_date").dt.day().alias("day"),
        pl.col("obs_date").dt.strftime(format="%m-%d").alias("clim_day"),
    )
    .group_by("month", "day", "clim_day")
    .agg(
        n_rows=pl.col(param).filter(is_ref).is_not_null().sum() + pl.col(param).filter(is_ref).is_null().sum(),
        n_values=pl.col(param).filter(is_ref).count(),
        norm=pl.col(param).filter(is_ref).mean(),
        std_norm=pl.col(param).filter(is_ref).std(),
        min_value=pl.col(param).min(),
        max_value=pl.col(param).max(),
    )
    .sort(["month", "day"])
)

print(lf_norm_test.explain(optimized=True))

df_norm_test = lf_norm_test.collect()

with pl.Config(tbl_cols=-1, tbl_rows=100, tbl_width_chars=200):
    print(df_norm_test)

SORT BY [col("month"), col("day")]
  AGGREGATE[maintain_order: false]
    [(col("tx").filter(col("__POLARS_CSER_0x100000007")).is_not_null().sum() + col("tx").filter(col("__POLARS_CSER_0x100000007")).null_count()).alias("n_rows"), col("tx").filter(col("__POLARS_CSER_0x100000007")).count().alias("n_values"), col("tx").filter(col("__POLARS_CSER_0x100000007")).mean().alias("norm"), col("tx").filter(col("__POLARS_CSER_0x100000007")).std().alias("std_norm"), col("tx").min().alias("min_value"), col("tx").max().alias("max_value")] BY [col("month"), col("day"), col("clim_day")]
    FROM
     WITH_COLUMNS:
     [col("obs_date").is_between([1991-01-01, 2020-12-31]).alias("__POLARS_CSER_0x100000007")] 
      simple π 5/5 ["month", "day", "clim_day", ... 2 other columns]
         WITH_COLUMNS:
         [col("obs_date").dt.month().alias("month"), col("obs_date").dt.day().alias("day"), col("obs_date").dt.to_string().alias("clim_day")] 
          simple π 2/2 ["obs_date", "tx"]
            Parquet SC

In [70]:
from statsmodels.nonparametric.smoothers_lowess import lowess
import numpy as np


def circular_rolling_mean(values, window_size):
    """
    Moyenne glissante circulaire centrée.
    
    window_size doit être impair.
    Exemple : 31 = 15 jours avant + jour courant + 15 jours après.
    """
    if window_size % 2 == 0:
        raise ValueError("window_size doit être impair pour une fenêtre centrée.")
    
    half = window_size // 2
    
    # On colle la fin au début et le début à la fin pour gérer la circularité.
    padded = np.concatenate([
        values[-half:],
        values,
        values[:half],
    ])

    # print("padded")
    # print(padded.size)
    # print(padded)
    
    windows = np.lib.stride_tricks.sliding_window_view(
        padded,
        window_shape=window_size,
    )

    # print("windows")
    # print(len(windows))
    # print(windows)

    # print(np.nanmean(windows))
    
    # nanmean permet de survivre si un jour a une valeur manquante.
    return np.nanmean(windows, axis=1)


def circular_lowess_span(values, span_days=31, it=0):
    n = len(values)
    x = np.arange(n)

    x_ext = np.concatenate([x - n, x, x + n])
    y_ext = np.concatenate([values, values, values])

    frac = span_days / len(y_ext)

    smooth_ext = lowess(
        endog=y_ext,
        exog=x_ext,
        frac=frac,
        it=it,
        return_sorted=False,
    )

    return smooth_ext[n:2*n]


def roughness_second_diff(y):
    y_ext = np.r_[y[-1], y, y[0]]
    second_diff = np.diff(y_ext, n=2)
    return np.sqrt(np.mean(second_diff ** 2))


def count_turning_points(y, eps=0.01):
    """
    Compte les changements de signe de pente.
    eps ignore les variations très petites.
    """
    diff = np.diff(np.r_[y, y[0]])

    # On neutralise les très petites variations.
    signs = np.where(np.abs(diff) < eps, 0, np.sign(diff))

    # On propage le dernier signe non nul pour éviter les faux changements dus aux plateaux.
    cleaned = []
    last = 0
    for s in signs:
        if s != 0:
            last = s
        cleaned.append(last)

    cleaned = np.array(cleaned)

    # On compte les changements + vers - ou - vers +.
    changes = cleaned != np.roll(cleaned, 1)

    return int(np.sum(changes))

In [71]:
candidate_windows = [31, 33, 35, 37, 39, 41, 43, 45]

raw = df_norm_test["norm"].to_numpy()

rows = []

for w in candidate_windows:
    # Rolling
    smooth_roll = circular_rolling_mean(raw, w)
    resid_roll = raw - smooth_roll

    rows.append({
        "method": "rolling",
        "window": w,
        "mae_vs_raw": float(np.mean(np.abs(resid_roll))),
        "rmse_vs_raw": float(np.sqrt(np.mean(resid_roll ** 2))),
        "max_abs_diff_vs_raw": float(np.max(np.abs(resid_roll))),
        "roughness": float(roughness_second_diff(smooth_roll)),
        "seasonal_amplitude": float(np.max(smooth_roll) - np.min(smooth_roll)),
        "turning_points": count_turning_points(smooth_roll, eps=0.0005),
    })

    # LOESS
    smooth_loess = circular_lowess_span(raw, span_days=w)
    resid_loess = raw - smooth_loess

    rows.append({
        "method": "loess",
        "window": w,
        "mae_vs_raw": float(np.mean(np.abs(resid_loess))),
        "rmse_vs_raw": float(np.sqrt(np.mean(resid_loess ** 2))),
        "max_abs_diff_vs_raw": float(np.max(np.abs(resid_loess))),
        "roughness": float(roughness_second_diff(smooth_loess)),
        "seasonal_amplitude": float(np.max(smooth_loess) - np.min(smooth_loess)),
        "turning_points": count_turning_points(smooth_loess, eps=0.0005),
    })

scores_df = pl.DataFrame(rows).sort(["method", "window"])
with pl.Config(tbl_cols=-1, tbl_rows=100, tbl_width_chars=200):
    print(scores_df)

shape: (16, 8)
┌─────────┬────────┬────────────┬─────────────┬─────────────────────┬───────────┬────────────────────┬────────────────┐
│ method  ┆ window ┆ mae_vs_raw ┆ rmse_vs_raw ┆ max_abs_diff_vs_raw ┆ roughness ┆ seasonal_amplitude ┆ turning_points │
│ ---     ┆ ---    ┆ ---        ┆ ---         ┆ ---                 ┆ ---       ┆ ---                ┆ ---            │
│ str     ┆ i64    ┆ f64        ┆ f64         ┆ f64                 ┆ f64       ┆ f64                ┆ i64            │
╞═════════╪════════╪════════════╪═════════════╪═════════════════════╪═══════════╪════════════════════╪════════════════╡
│ loess   ┆ 31     ┆ 0.437528   ┆ 0.53436     ┆ 1.367161            ┆ 0.007946  ┆ 21.251286          ┆ 4              │
│ loess   ┆ 33     ┆ 0.447018   ┆ 0.545962    ┆ 1.391322            ┆ 0.00711   ┆ 21.198383          ┆ 2              │
│ loess   ┆ 35     ┆ 0.455714   ┆ 0.55667     ┆ 1.42989             ┆ 0.006486  ┆ 21.141167          ┆ 2              │
│ loess   ┆ 37     ┆ 0.46

In [74]:
print(circular_lowess_span(raw, span_days=45))

[ 8.57660827  8.5885049   8.6032034   8.62182844  8.64517375  8.6736442
  8.7076415   8.74754523  8.79356545  8.8456395   8.90336337  8.96593749
  9.03184405  9.09949648  9.16740256  9.23441876  9.30003213  9.36456538
  9.42868033  9.49319902  9.55876222  9.6256792   9.69424278  9.76451924
  9.83650766  9.91005079  9.98501851 10.06104676 10.1376517  10.21424933
 10.29039073 10.36587044 10.4405078  10.51391685 10.58573374 10.65594803
 10.72468231 10.79194455 10.8579285  10.92286241 10.98689773 11.05042901
 11.11374992 11.17690053 11.23965634 11.3018136  11.36348921 11.42519493
 11.48796786 11.55337443 11.62334745 11.69964717 11.78359579 11.87609466
 11.97764746 12.08854564 12.20852112 12.33716248 12.4738762  12.61753567
 12.76652617 12.9189995  13.07312642 13.22704914 13.37883595 13.52667594
 13.669426   13.80645767 13.93756712 14.06317188 14.18406964 14.30067992
 14.41342013 14.52266586 14.62848354 14.73100715 14.83039407 14.92641628
 15.01830668 15.1051697  15.18613126 15.26020534 15.

In [76]:
df_norm_test_smoothed = df_norm_test.with_columns(
    pl.Series("lowess_45d", circular_lowess_span(raw, span_days=45).tolist())
)

with pl.Config(tbl_cols=-1, tbl_rows=400, tbl_width_chars=200):
    print(df_norm_test_smoothed)

shape: (366, 10)
┌───────┬─────┬──────────┬────────┬──────────┬───────────┬──────────┬───────────┬───────────┬────────────┐
│ month ┆ day ┆ clim_day ┆ n_rows ┆ n_values ┆ norm      ┆ std_norm ┆ min_value ┆ max_value ┆ lowess_45d │
│ ---   ┆ --- ┆ ---      ┆ ---    ┆ ---      ┆ ---       ┆ ---      ┆ ---       ┆ ---       ┆ ---        │
│ i8    ┆ i8  ┆ str      ┆ u32    ┆ u32      ┆ f64       ┆ f64      ┆ f64       ┆ f64       ┆ f64        │
╞═══════╪═════╪══════════╪════════╪══════════╪═══════════╪══════════╪═══════════╪═══════════╪════════════╡
│ 1     ┆ 1   ┆ 01-01    ┆ 30     ┆ 30       ┆ 8.396667  ┆ 3.01599  ┆ -1.5      ┆ 14.9      ┆ 8.576608   │
│ 1     ┆ 2   ┆ 01-02    ┆ 30     ┆ 30       ┆ 7.603333  ┆ 3.233391 ┆ -1.0      ┆ 19.0      ┆ 8.588505   │
│ 1     ┆ 3   ┆ 01-03    ┆ 30     ┆ 30       ┆ 9.15      ┆ 3.559422 ┆ 0.6       ┆ 16.5      ┆ 8.603203   │
│ 1     ┆ 4   ┆ 01-04    ┆ 30     ┆ 30       ┆ 9.11      ┆ 4.528861 ┆ 0.9       ┆ 18.7      ┆ 8.621828   │
│ 1     ┆ 5   ┆ 01-0